# Capstone Project: Machine Learning Models

**Instructor:** Abishek Ganesh

**Your Name:** Kalpana   
**Your Role:** Data Scientist  
**Client:** Smart city platform  
**Due Date:** May10, 2026

---


### How to Verify Your Work
- **Assert Tests**: I have provided `assert` statements to help you check your work. Your code should pass all of them.
- **Example Outputs**: Each section shows expected outputs. Compare yours to make sure you're on track.
- **Written Reflections**: Answer all reflection questions - they help me understand your thinking.

---

## Setup: Import Libraries

Run this cell first to import all the libraries you'll need.

In [5]:
# Required imports - run this cell first!
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Regression
from sklearn.linear_model import LinearRegression

# Classification
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Evaluation
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc
)

# Display settings
pd.set_option('display.max_columns', None)
%matplotlib inline

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


### Task: Load the Customer Data

**Your Task:** Create a function that loads the city_traffic_data, urbanpulse_311_complaints and returns a DataFrame.

### Example Verification
```python
df1 = load_customer_data('city_traffic_accidents.csv')


print(df.shape)  # Should print (100000, 17)


In [6]:
import pandas as pd

df1 = pd.read_csv('../data/raw/city_traffic_accidents.csv')
df1.head(10)

In [7]:
import pandas as pd

def load_customer_data(filepath):
    """
    Load customer data from a CSV file.
    
    Args:
        filepath (str): Path to the CSV file
        
    Returns:
        pd.DataFrame: The loaded data
    """
    # Your code here
    data = pd.read_csv(filepath)
    return data

In [8]:
import pandas as pd

df1 = pd.read_csv('../data/raw/city_traffic_accidents.csv')
df1.head(10)

In [ ]:
# ============================================================
# ⚠️  DO NOT MODIFY THIS CELL - VERIFICATION TESTS ONLY  ⚠️
# Just run this cell to check if your code is working.
# Hidden test cases depend on this exact structure.
# ============================================================

# Load the data
df1 = load_customer_data('city_traffic_accidents.csv')

# Basic Verification
assert df1 is not None, "Function returned None - did you forget to return the DataFrame?"
assert isinstance(df1, pd.DataFrame), "Function should return a pandas DataFrame"
assert df1.shape == (500000, 46), f"Expected shape (500000, 46), got {df1.shape}"
assert 'Severity' in df1.columns, "Missing regression target 'Severity'"
assert 'Severity' in df1.columns, "Missing classification target 'Severity'"
print("[PASS] Part 1 Tests Passed!")
print(f"\nDataset loaded: {df1.shape[0]:,} customers, {df1.shape[1]} columns")

[PASS] Part 1 Tests Passed!

Dataset loaded: 500,000 customers, 46 columns


In [ ]:
# Check the data types and missing values
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 46 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ID                     500000 non-null  str    
 1   Source                 500000 non-null  str    
 2   Severity               500000 non-null  int64  
 3   Start_Time             500000 non-null  str    
 4   End_Time               500000 non-null  str    
 5   Start_Lat              500000 non-null  float64
 6   Start_Lng              500000 non-null  float64
 7   End_Lat                280022 non-null  float64
 8   End_Lng                280022 non-null  float64
 9   Distance(mi)           500000 non-null  float64
 10  Description            500000 non-null  str    
 11  Street                 499346 non-null  str    
 12  City                   499972 non-null  str    
 13  County                 500000 non-null  str    
 14  State                  500000 non-null  str    

---

## Part 2: Data Preparation 

**The Problem:** Before training any models, we need to:
1. Handle missing values
2. Encode categorical variables
3. Scale numeric features (important for Logistic Regression)
4. Split our data into training and test sets

**Note:** You can reference your Unit 2 capstone code for handling missing values, encoding, and scaling - the techniques are the same!

### Task 2a: Prepare Features

Create a function that:

Steps:
Load the city traffic accidents dataset.
    key gotchas:
    - Severity is 1-4 scale measuring TRAFFIC IMPACT, not injury severity

    - Severity 2 dominates (~80% of records) — MAJOR class imbalance

      A naive model that predicts Severity 2 for everything gets ~80% accuracy

      but is completely useless. You MUST use class weights or SMOTE.

      Weighted F1 is the real metric, not accuracy.

    - Datetime columns need parsing

    - Weather columns have significant missing values
    
    - Some lat/lng values are null (End_Lat, End_Lng especially)
    ""


In [ ]:

def prepare_features(df):
    """
    Prepare features from city_traffic_accidents.csv for machine learning.

    Steps:
    1. Drop non-predictive / high-cardinality / high-missing columns
    2. Fill missing numeric values with median
    3. Convert boolean columns to int (0/1)
    4. Label encode Weather_Condition (106 unique values)
    5. One-hot encode low-cardinality categoricals
    6. Scale numeric features using StandardScaler

    Args:
        df (pd.DataFrame): The raw DataFrame

    Returns:
        pd.DataFrame: The prepared DataFrame with all numeric columns (scaled)
    """
    from sklearn.preprocessing import StandardScaler, LabelEncoder

    # Make a copy so we don't modify the original
    df_prep = df.copy()

    # Your code here
    
    return df_prep

In [ ]:
def prepare_features(df):
    """
    Prepare features from city_traffic_accidents.csv for machine learning.
    
    Steps:
    1. Remove duplicate rows based on 'ID' column
    2. Drop non-predictive / high-cardinality columns
    3. Fill missing numeric values with median
    4. Convert boolean columns to int (0/1)
    5. Label encode Weather_Condition (106 unique values)
    6. One-hot encode low-cardinality categoricals
    7. Scale numeric features using StandardScaler
    
    Args:
        df (pd.DataFrame): The raw DataFrame
        
    Returns:
        pd.DataFrame: The prepared DataFrame with all numeric columns (scaled)
    """
    from sklearn.preprocessing import StandardScaler, LabelEncoder

    # Make a copy so we don't modify the original
    df_prep = df.copy()

    # Step 1: Remove duplicate rows based on 'ID' column
    before = len(df_prep)
    df_prep = df_prep.drop_duplicates(subset='ID', keep='first')
    removed = before - len(df_prep)
    if removed:
        print(f"  Removed {removed:,} duplicate rows (kept first occurrence)")

    # Step 2: Drop non-predictive / high-cardinality columns
    drop_cols = [
        'ID', 'Description', 'Street', 'City', 'County',
        'Zipcode', 'Airport_Code', 'Country',
        'Start_Time', 'End_Time', 'Weather_Timestamp',
        'End_Lat', 'End_Lng',  # ~44% missing
        'Turning_Loop',          # near-zero variance
    ]
    df_prep = df_prep.drop(columns=drop_cols, errors='ignore')

    # Step 3: Fill missing numeric values with median
    numeric_cols = df_prep.select_dtypes(include='number').columns
    for col in numeric_cols:
        df_prep[col] = df_prep[col].fillna(df_prep[col].median())

    # Step 4: Convert boolean columns to int (0/1)
    bool_cols = df_prep.select_dtypes(include='bool').columns
    df_prep[bool_cols] = df_prep[bool_cols].astype(int)

    # Step 5: Label encode Weather_Condition (106 unique values - too many to one-hot)
    le = LabelEncoder()
    df_prep['Weather_Condition'] = df_prep['Weather_Condition'].fillna('Unknown')
    df_prep['Weather_Condition'] = le.fit_transform(df_prep['Weather_Condition'])

    # Step 6: One-hot encode low-cardinality categoricals
    low_card_cats = ['Source', 'State', 'Timezone', 'Wind_Direction',
                     'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight',
                     'Astronomical_Twilight']
    for col in low_card_cats:
        if col in df_prep.columns:
            df_prep[col] = df_prep[col].fillna('Unknown')
    df_prep = pd.get_dummies(df_prep, columns=[c for c in low_card_cats if c in df_prep.columns], drop_first=True)

    # Step 7: Scale numeric features (exclude Severity and binary 0/1 columns)
    bool_like_cols = [c for c in df_prep.select_dtypes(include='number').columns
                      if df_prep[c].nunique() <= 2]
    scale_cols = [c for c in df_prep.select_dtypes(include='number').columns
                  if c != 'Severity' and c not in bool_like_cols]
    scaler = StandardScaler()
    df_prep[scale_cols] = scaler.fit_transform(df_prep[scale_cols])

    return df_prep


In [ ]:
# Prepare the features
df_prepared = prepare_features(df1)

# Basic Verification
assert df_prepared is not None, "Function returned None - did you forget to return the DataFrame?"
assert isinstance(df_prepared, pd.DataFrame), "Function should return a pandas DataFrame"
assert df_prepared.shape[0] == 500000, f"Expected 500000 rows, got {df_prepared.shape[0]}"
assert df_prepared.shape[0] == df1.shape[0], f"Row count changed - expected {df1.shape[0]}, got {df_prepared.shape[0]}"
assert 'Severity' in df_prepared.columns, "Missing target column 'Severity'"
assert 'ID' not in df_prepared.columns, "Column 'ID' should have been dropped"
assert 'Description' not in df_prepared.columns, "Column 'Description' should have been dropped"
assert 'End_Lat' not in df_prepared.columns, "Column 'End_Lat' should have been dropped"
assert 'End_Lng' not in df_prepared.columns, "Column 'End_Lng' should have been dropped"
assert 'Start_Time' not in df_prepared.columns, "Column 'Start_Time' should have been dropped"
assert df_prepared.isnull().sum().sum() == 0, f"Expected 0 missing values, got {df_prepared.isnull().sum().sum()}"
assert df_prepared.select_dtypes(include='object').shape[1] == 0, "All columns should be numeric after encoding"
assert 'Weather_Condition' in df_prepared.columns, "Weather_Condition should be label-encoded (not dropped)"
assert df_prepared['Weather_Condition'].dtype in ['int64', 'int32', 'float64'], "Weather_Condition should be numeric"
assert df_prepared['Junction'].isin([0, 1]).all(), "Junction should be 0 or 1"
assert df_prepared['Crossing'].isin([0, 1]).all(), "Crossing should be 0 or 1"
assert df_prepared['Traffic_Signal'].isin([0, 1]).all(), "Traffic_Signal should be 0 or 1"
assert df_prepared['Severity'].isin([1, 2, 3, 4]).all(), "Severity should only contain values 1, 2, 3, or 4"
assert any(col.startswith('State_') for col in df_prepared.columns), "State should be one-hot encoded"
assert any(col.startswith('Source_') for col in df_prepared.columns), "Source should be one-hot encoded"
assert any(col.startswith('Wind_Direction_') for col in df_prepared.columns), "Wind_Direction should be one-hot encoded"
print("[PASS] Part 2a Preparation Tests Passed!")
print(f"\nPrepared DataFrame: {df_prepared.shape[0]:,} rows, {df_prepared.shape[1]} columns")
print(f"Missing values: {df_prepared.isnull().sum().sum()}")


[PASS] Part 2a Preparation Tests Passed!

Prepared DataFrame: 500,000 rows, 110 columns
Missing values: 0


In [ ]:
df_prepared = prepare_features(df1)
df_prepared.head(10)

,Severity,Start_Lat,Start_Lng,Distance(mi),Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Source_Source2,Source_Source3,State_AR,State_AZ,State_CA,State_CO,State_CT,State_DC,State_DE,State_FL,State_GA,State_IA,State_ID,State_IL,State_IN,State_KS,State_KY,State_LA,State_MA,State_MD,State_ME,State_MI,State_MN,State_MO,State_MS,State_MT,State_NC,State_ND,State_NE,State_NH,State_NJ,State_NM,State_NV,State_NY,State_OH,State_OK,State_OR,State_PA,State_RI,State_SC,State_SD,State_TN,State_TX,State_UT,State_VA,State_VT,State_WA,State_WI,State_WV,State_WY,Timezone_US/Eastern,Timezone_US/Mountain,Timezone_US/Pacific,Timezone_Unknown,Wind_Direction_Calm,Wind_Direction_E,Wind_Direction_ENE,Wind_Direction_ESE,Wind_Direction_East,Wind_Direction_N,Wind_Direction_NE,Wind_Direction_NNE,Wind_Direction_NNW,Wind_Direction_NW,Wind_Direction_North,Wind_Direction_S,Wind_Direction_SE,Wind_Direction_SSE,Wind_Direction_SSW,Wind_Direction_SW,Wind_Direction_South,Wind_Direction_Unknown,Wind_Direction_VAR,Wind_Direction_Variable,Wind_Direction_W,Wind_Direction_WNW,Wind_Direction_WSW,Wind_Direction_West,Sunrise_Sunset_Night,Sunrise_Sunset_Unknown,Civil_Twilight_Night,Civil_Twilight_Unknown,Nautical_Twilight_Night,Nautical_Twilight_Unknown,Astronomical_Twilight_Night,Astronomical_Twilight_Unknown
0,2,-0.073010,0.924328,-0.324662,0.387666,0.506510,0.360523,0.227519,0.332156,-0.518556,-0.067044,0.989264,0,0,0,0,0,0,0,0,0,0,0,0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,2,-0.022944,-0.075065,-0.324662,0.759777,0.868288,0.891538,-0.533144,0.332156,0.071103,0.045776,0.448353,0,0,0,0,0,0,0,0,0,0,0,0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,True,False,False,False
2,2,-0.525612,0.455159,-0.236280,1.450841,1.540162,-0.834263,-0.212865,0.332156,-0.911662,-0.067044,-0.705591,0,0,0,0,1,0,0,0,0,0,0,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
3,2,-0.420294,-1.317670,0.180624,1.185047,1.281749,-1.453781,-0.703293,0.332156,0.267657,-0.067044,-0.705591,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,Fals

In [ ]:
def load_accidents(filepath: str) -> pd.DataFrame:
    """
    Load the city traffic accidents dataset.
    Key gotchas:
    - Severity is 1-4 scale measuring TRAFFIC IMPACT, not injury severity
    - Severity 2 dominates (~80% of records) — MAJOR class imbalance
      A naive model that predicts Severity 2 for everything gets ~80% accuracy
      but is completely useless. You MUST use class weights or SMOTE.
      Weighted F1 is the real metric, not accuracy.
    - Datetime columns need parsing
    - Weather columns have significant missing values
    - Some lat/lng values are null (End_Lat, End_Lng especially)
    """
    df = pd.read_csv(filepath, low_memory=False)

    # Parse datetime columns
    for col in ['Start_Time', 'End_Time', 'Weather_Timestamp']:
        df[col] = pd.to_datetime(df[col], errors='coerce')

    # Derived time features (useful for modeling)
    df['Hour']       = df['Start_Time'].dt.hour
    df['DayOfWeek']  = df['Start_Time'].dt.dayofweek   # 0=Mon, 6=Sun
    df['Month']      = df['Start_Time'].dt.month
    df['Duration_min'] = (
        (df['End_Time'] - df['Start_Time'])
        .dt.total_seconds()
        .div(60)
        .clip(lower=0)   # drop negative durations from data errors
    )

    # Weather columns: ~44% of End_Lat/Lng missing — expected, keep as-is
    # Wind_Chill and Precipitation have high missingness; flag rather than impute
    df['Wind_Chill_missing']    = df['Wind_Chill(F)'].isna().astype(int)
    df['Precipitation_missing'] = df['Precipitation(in)'].isna().astype(int)

    # Zipcode loaded as float due to nulls — cast to nullable int-string
    df['Zipcode'] = df['Zipcode'].astype(str).str.extract(r'^(\d{5})')[0]


    # Severity reminder: do NOT treat as continuous — it's ordinal/categorical
    df['Severity'] = df['Severity'].astype('category')

    return df


In [ ]:
# Basic Verification
df_accidents = load_accidents('city_traffic_accidents.csv')

assert df_accidents is not None, "Function returned None"
assert isinstance(df_accidents, pd.DataFrame), "Should return a DataFrame"
assert df_accidents.shape == (500000, 52), f"Expected (500000, 52), got {df_accidents.shape}"
assert pd.api.types.is_datetime64_any_dtype(df_accidents['Start_Time']), "Start_Time should be datetime"
assert pd.api.types.is_datetime64_any_dtype(df_accidents['End_Time']), "End_Time should be datetime"

print(f"Shape: {df_accidents.shape}")
print("All checks passed!")

Shape: (500000, 52)
All checks passed!


In [ ]:
df_accidents.head()

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,Street,City,County,State,Zipcode,Country,Timezone,Airport_Code,Weather_Timestamp,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight,Hour,DayOfWeek,Month,Duration_min,Wind_Chill_missing,Precipitation_missing
0,A-1784167,Source2,2,2019-10-29 13:16:54,2019-10-29 15:21:34,35.834797,-78.638512,NaN,NaN,0.000,Accident on Six Forks Rd at I-440 Cliff Benson...,I-440 W,Raleigh,Wake,NC,27609,US,US/Eastern,KRDU,2019-10-29 12:51:00,69.0,69.0,73.0,29.77,10.0,E,5.0,0.00,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,13.0,1.0,10.0,124.666667,0,0
1,A-862811,Source2,2,2021-10-13 06:30:00,2021-10-13 06:59:15,36.088970,-96.011734,NaN,NaN,0.000,Accident on I-44 Eastbound at Union Ave.,I-44 W,Tulsa,Tulsa,OK,74107,US,US/Central,KRVS,2021-10-13 06:29:00,76.0,76.0,85.0,29.01,10.0,S,8.0,0.01,Light Rain with Thunder,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Night,Night,Day,6.0,2.0,10.0,29.250000,0,0
2,A-4054572,Source1,2,2022-08-14 14:42:58,2022-08-14 16:27:58,33.537049,-86.794445,33.535373,-86.796156,0.152,Incident on I-20 WB near MM 126 Road closed. T...,I-20 W,Birmingham,Jefferson,AL,35234,US,US/Central,KBHM,2022-08-14 14:53:00,89.0,89.0,46.0,29.33,10.0,VAR,3.0,0.00,Fair,False,False,False,False,True,False,False,False,False,False,False,False,False,Day,Day,Day,Day,14.0,6.0,8.0,105.000000,0,0
3,A-6147589,Source1,2,2021-06-25 19:13:44,2021-06-25 20:42:30,34.071722,-117.612886,34.078917,-117.625339,0.869,Slow traffic on I-10 W - San Bernardino Fwy W ...,I-10 W,Ontario,San Bernardino,CA,91764,US,US/Pacific,KONT,2021-06-25 18:53:00,84.0,84.0,32.0,28.84,10.0,W,9.0,0.00,Fair,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,19.0,4.0,6.0,88.766667,0,0
4,A-5025169,Source1,2,2022-03-18 12:50:30,2022-03-18 13:13:00,40.324235,-76.790464,40.322625,-76.788114,0.166,Stationary traffic on US-22 E from I-81 N to S...,Ridgeview Dr,Harrisburg,Dauphin,PA,17112,US,US/Eastern,KCXY,2022-03-18 12:56:00,71.0,71.0,47.0,29.63,10.0,ENE,5.0,0.00,Fair,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,12.0,4.0,3.0,22.500000,0,0


In [ ]:
df_accidents.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 52 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ID                     500000 non-null  str           
 1   Source                 500000 non-null  str           
 2   Severity               500000 non-null  category      
 3   Start_Time             452263 non-null  datetime64[us]
 4   End_Time               452263 non-null  datetime64[us]
 5   Start_Lat              500000 non-null  float64       
 6   Start_Lng              500000 non-null  float64       
 7   End_Lat                280022 non-null  float64       
 8   End_Lng                280022 non-null  float64       
 9   Distance(mi)           500000 non-null  float64       
 10  Description            500000 non-null  str           
 11  Street                 499346 non-null  str           
 12  City                   499972 non-null  str           


In [ ]:
#df create_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
def create_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Time patterns are among the strongest predictors of accident severity.

    Features to extract:
    - Hour of day (rush hour vs. off-peak)
    - Day of week (weekday vs. weekend)
    - Month (seasonal patterns — winter ice, summer heat)
    - Duration of traffic impact
    - Is it dark? (Sunrise_Sunset column helps, but you can derive from time too)
    """
    # Your code here
    df['hour'] = df['Start_Time'].dt.hour
    df['day_of_week'] = df['Start_Time'].dt.dayofweek
    df['month'] = df['Start_Time'].dt.month
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

    # Rush hour flags
    df['is_morning_rush'] = df['hour'].between(7, 9).astype(int)
    df['is_evening_rush'] = df['hour'].between(16, 19).astype(int)
    df['is_rush_hour'] = (df['is_morning_rush'] | df['is_evening_rush']).astype(int)

    # Duration of traffic impact (in minutes)
    if 'End_Time' in df.columns:
        df['duration_min'] = (df['End_Time'] - df['Start_Time']).dt.total_seconds() / 60
        # Cap extreme values
        df['duration_min'] = df['duration_min'].clip(0, 1440)  # Max 24 hours

    return df


In [ ]:
# Test create_temporal_features
df_test = df_accidents.copy()
df_test = create_temporal_features(df_test)

# Verify temporal features were created
assert 'hour' in df_test.columns, "Missing 'hour' column"
assert 'day_of_week' in df_test.columns, "Missing 'day_of_week' column"
assert 'month' in df_test.columns, "Missing 'month' column"
assert 'is_weekend' in df_test.columns, "Missing 'is_weekend' column"
assert 'is_morning_rush' in df_test.columns, "Missing 'is_morning_rush' column"
assert 'is_evening_rush' in df_test.columns, "Missing 'is_evening_rush' column"
assert 'is_rush_hour' in df_test.columns, "Missing 'is_rush_hour' column"
assert 'duration_min' in df_test.columns, "Missing 'duration_min' column"

# Validate hour values (0-23)
assert df_test['hour'].min() >= 0 and df_test['hour'].max() <= 23, "Hour values out of range"

# Validate day_of_week values (0-6)
assert df_test['day_of_week'].min() >= 0 and df_test['day_of_week'].max() <= 6, "Day of week values out of range"

# Validate month values (1-12)
assert df_test['month'].min() >= 1 and df_test['month'].max() <= 12, "Month values out of range"

# Validate binary flags (0 or 1)
for col in ['is_weekend', 'is_morning_rush', 'is_evening_rush', 'is_rush_hour']:
    assert set(df_test[col].unique()).issubset({0, 1}), f"{col} should only contain 0 or 1"

# Validate duration (0-1440 minutes, i.e., 0-24 hours)
assert df_test['duration_min'].min() >= 0 and df_test['duration_min'].max() <= 1440, "Duration values out of range"

print("✓ All temporal features created successfully")
print(f"  - Temporal columns added: {len(df_test.columns) - len(df_accidents.columns)}")
print(f"  - DataFrame shape: {df_test.shape}")
print(f"  - Sample hour values: {df_test['hour'].value_counts().head(3).to_dict()}")
print(f"  - Rush hour stats: {df_test['is_rush_hour'].sum()} records during rush hours")

✓ All temporal features created successfully
  - Temporal columns added: 8
  - DataFrame shape: (500000, 60)
  - Sample hour values: {7.0: 35227, 8.0: 35162, 16.0: 33581}
  - Rush hour stats: 201514 records during rush hours


In [ ]:
#def process_weather_features(df: pd.DataFrame) -> pd.DataFrame:
def process_weather_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Weather is a major factor in accident severity.

    Missing values in weather columns are NOT random — they often mean:
    - Weather station was offline
    - Data wasn't available at the time of the accident
    - The weather API didn't return data for that location

    Strategy: Create a "weather_data_available" flag, then impute or drop.

    Key weather features:
    - Temperature(F): Freezing conditions are dangerous
    - Visibility(mi): Low visibility = more severe accidents
    - Precipitation(in): Rain/snow increases severity
    - Weather_Condition: Categorical (Clear, Rain, Snow, Fog, etc.)
    """
     # Your code here

    weather_cols = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)',
                    'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']

    # Flag for whether weather data is available
    
    df['weather_data_available'] = df[weather_cols].notna().all(axis=1).astype(int)

    # Freezing conditions
    if 'Temperature(F)' in df.columns:
        df['is_freezing'] = (df['Temperature(F)'] <= 32).astype(int)

    # Low visibility
    if 'Visibility(mi)' in df.columns:
        df['low_visibility'] = (df['Visibility(mi)'] < 2).astype(int)

    # Group weather conditions
    if 'Weather_Condition' in df.columns:
        df['weather_group'] = df['Weather_Condition'].apply(categorize_weather)

    return df


def categorize_weather(condition) -> str:
    """Group detailed weather conditions into broader categories."""
    if pd.isna(condition):
        return 'unknown'

    condition = str(condition).lower()

    if any(w in condition for w in ['clear', 'fair']):
        return 'clear'
    elif any(w in condition for w in ['cloud', 'overcast']):
        return 'cloudy'
    elif any(w in condition for w in ['rain', 'drizzle', 'shower']):
        return 'rain'
    elif any(w in condition for w in ['snow', 'sleet', 'ice', 'wintry']):
        return 'snow_ice'
    elif any(w in condition for w in ['fog', 'mist', 'haze', 'smoke']):
        return 'low_visibility'
    elif any(w in condition for w in ['thunder', 'storm']):
        return 'storm'
    else:
        return 'other'


In [ ]:
# Test process_weather_features
df_weather_test = df_test.copy()  # Use the df_test from temporal features
df_weather_test = process_weather_features(df_weather_test)

# Verify weather features were created
assert 'weather_data_available' in df_weather_test.columns, "Missing 'weather_data_available' column"
assert 'is_freezing' in df_weather_test.columns, "Missing 'is_freezing' column"
assert 'low_visibility' in df_weather_test.columns, "Missing 'low_visibility' column"
assert 'weather_group' in df_weather_test.columns, "Missing 'weather_group' column"

# Validate binary flags (0 or 1)
for col in ['weather_data_available', 'is_freezing', 'low_visibility']:
    assert set(df_weather_test[col].unique()).issubset({0, 1}), f"{col} should only contain 0 or 1"

# Validate weather_group categories
valid_weather_groups = {'clear', 'cloudy', 'rain', 'snow_ice', 'low_visibility', 'storm', 'other', 'unknown'}
actual_groups = set(df_weather_test['weather_group'].unique())
assert actual_groups.issubset(valid_weather_groups), f"Invalid weather groups: {actual_groups - valid_weather_groups}"

# Data quality checks
weather_data_pct = df_weather_test['weather_data_available'].mean() * 100
freezing_records = df_weather_test['is_freezing'].sum()
low_vis_records = df_weather_test['low_visibility'].sum()

print("✓ All weather features created and validated")
print(f"  - Weather data available: {weather_data_pct:.1f}% of records")
print(f"  - Freezing conditions: {freezing_records:,} records")
print(f"  - Low visibility: {low_vis_records:,} records")
print(f"  - Weather group distribution:")
print(f"    {df_weather_test['weather_group'].value_counts().to_dict()}")

✓ All weather features created and validated
  - Weather data available: 69.9% of records
  - Freezing conditions: 39,888 records
  - Low visibility: 16,043 records
  - Weather group distribution:
    {'clear': 220638, 'cloudy': 204121, 'rain': 34888, 'low_visibility': 13084, 'snow_ice': 11377, 'unknown': 11300, 'storm': 4312, 'other': 280}


In [ ]:
def process_road_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    The dataset has 13 boolean road feature columns.

    These are already binary (True/False) and very useful for ML models.
    Consider creating aggregate features:
    - total_road_features: count of road features at the accident location
    - has_traffic_control: any of traffic signal, stop, give way, etc.
    """
     # Your code here
    road_features = ['Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction',
                     'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop',
                     'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop']

    existing = [f for f in road_features if f in df.columns]

    # Total road features present
    df['n_road_features'] = df[existing].sum(axis=1)

    # Traffic control present
    control_features = ['Traffic_Signal', 'Stop', 'Give_Way', 'Traffic_Calming']
    existing_control = [f for f in control_features if f in df.columns]
    df['has_traffic_control'] = df[existing_control].any(axis=1).astype(int)

    return df


In [ ]:
# Test process_road_features
df_road_test = df_weather_test.copy()  # Use the df_weather_test from previous features
df_road_test = process_road_features(df_road_test)

# Verify road features were created
assert 'n_road_features' in df_road_test.columns, "Missing 'n_road_features' column"
assert 'has_traffic_control' in df_road_test.columns, "Missing 'has_traffic_control' column"

# Validate n_road_features
assert df_road_test['n_road_features'].dtype in ['int64', 'int32', 'int'], f"n_road_features should be integer, got {df_road_test['n_road_features'].dtype}"
assert df_road_test['n_road_features'].min() >= 0, "n_road_features should not be negative"
assert df_road_test['n_road_features'].max() <= 13, f"n_road_features should not exceed 13, got max {df_road_test['n_road_features'].max()}"

# Validate has_traffic_control (binary flag)
assert set(df_road_test['has_traffic_control'].unique()).issubset({0, 1}), "has_traffic_control should only contain 0 or 1"

# Data quality checks
mean_road_features = df_road_test['n_road_features'].mean()
traffic_control_pct = df_road_test['has_traffic_control'].mean() * 100
locations_with_features = (df_road_test['n_road_features'] > 0).sum()

print("✓ All road features created and validated")
print(f"  - Average road features per location: {mean_road_features:.2f}")
print(f"  - Locations with any road features: {locations_with_features:,}")
print(f"  - Locations with traffic control: {traffic_control_pct:.1f}%")
print(f"  - Road feature distribution:")
print(f"    {df_road_test['n_road_features'].value_counts().sort_index().to_dict()}")

✓ All road features created and validated
  - Average road features per location: 0.42
  - Locations with any road features: 148,910
  - Locations with traffic control: 17.7%
  - Road feature distribution:
    {0: 351090, 1: 99837, 2: 39436, 3: 8053, 4: 1399, 5: 175, 6: 10}


In [ ]:
def analyze_severity_distribution(df: pd.DataFrame):
    """
    Severity distribution is heavily imbalanced:
    - Severity 1: ~1-2% (very rare)
    - Severity 2: ~80% (dominant — this is your biggest challenge)
    - Severity 3: ~12-15%
    - Severity 4: ~5-8%

    This is a MAJOR challenge. If you just predict class 2 for everything,
    you'll get ~80% accuracy but your model is COMPLETELY USELESS.
    Weighted F1 is the real evaluation metric, not accuracy.

    Strategies:
    1. Class weights: Give higher weight to minority classes
       - sklearn: class_weight='balanced'
       - TensorFlow/Keras: class_weight parameter in model.fit()
    2. SMOTE or oversampling for minority classes
    3. Undersampling the majority class (Severity 2)
    4. Consider binary: "severe" (3-4) vs "not severe" (1-2)
    5. Focal loss — designed for class imbalance

    For evaluation: Use weighted F1, not just accuracy.
    Weighted F1 accounts for class imbalance by weighting each class by its support.
    """
     # Your code here
     
    print("Severity Distribution:")
    print(df['Severity'].value_counts().sort_index())
    print(f"\nClass ratios:")
    print(df['Severity'].value_counts(normalize=True).sort_index().round(3))


In [ ]:
# Test analyze_severity_distribution
print("\n" + "="*60)
print("Analyzing Severity Distribution")
print("="*60)
analyze_severity_distribution(df_road_test)

# Verify severity data and check for class imbalance
severity_counts = df_road_test['Severity'].value_counts().sort_index()
severity_pcts = df_road_test['Severity'].value_counts(normalize=True).sort_index() * 100

# Check for expected imbalance pattern
assert len(severity_counts) > 0, "No severity data found"
assert 2 in severity_counts.index, "Severity 2 should be present (dominant class)"

# Verify the dominant class (Severity 2) is actually the largest
severity_2_pct = severity_pcts[2]
assert severity_2_pct > 50, f"Severity 2 should dominate (~80%), got {severity_2_pct:.1f}%"

print("\n✓ Severity distribution analysis complete")
print(f"  - Total records: {len(df_road_test):,}")
print(f"  - Classes present: {sorted(severity_counts.index.tolist())}")
print(f"  - Class imbalance detected: Severity 2 = {severity_2_pct:.1f}% (as expected)")
print(f"  - Minority classes (1,3,4) = {(100-severity_2_pct):.1f}%")
print(f"\n⚠️  IMPORTANT: Use Weighted F1, not accuracy! ~80% accuracy is useless here.")


Analyzing Severity Distribution
Severity Distribution:
Severity
1      4358
2    398335
3     84063
4     13244
Name: count, dtype: int64

Class ratios:
Severity
1    0.009
2    0.797
3    0.168
4    0.026
Name: proportion, dtype: float64

✓ Severity distribution analysis complete
  - Total records: 500,000
  - Classes present: [1, 2, 3, 4]
  - Class imbalance detected: Severity 2 = 79.7% (as expected)
  - Minority classes (1,3,4) = 20.3%

⚠️  IMPORTANT: Use Weighted F1, not accuracy! ~80% accuracy is useless here.


In [ ]:
def build_ml_ready_df(filepath: str) -> pd.DataFrame:
    """
    Build the final ML-ready DataFrame for city_traffic_accidents.csv.

    Combines all feature engineering steps into one pipeline:
      1. load_accidents       → parse datetimes, add missingness flags
      2. create_temporal_features → hour, day_of_week, rush-hour flags, duration
      3. process_weather_features → freezing/low-vis flags, weather_group
      4. process_road_features    → n_road_features, has_traffic_control
      5. Drop non-predictive / high-cardinality columns
      6. Fill remaining missing numerics with median
      7. Label-encode high-cardinality categorical (Weather_Condition, weather_group)
      8. One-hot encode low-cardinality categoricals
      9. Scale continuous numeric features with StandardScaler
         (Severity and binary 0/1 columns are excluded)

    Returns
    -------
    pd.DataFrame
        All columns numeric, no missing values, Severity intact (int 1-4).
    """
    from sklearn.preprocessing import StandardScaler, LabelEncoder

    # ── Step 1-4: feature engineering ────────────────────────────────────────
    df = load_accidents(filepath)
    df = create_temporal_features(df)
    df = process_weather_features(df)
    df = process_road_features(df)

    # ── Step 5: drop non-predictive / high-cardinality columns ───────────────
    drop_cols = [
        # identifiers / free text
        'ID', 'Description', 'Street', 'City', 'County',
        'Zipcode', 'Airport_Code',
        # raw datetime objects (temporal features already extracted)
        'Start_Time', 'End_Time', 'Weather_Timestamp',
        # location coords with ~44 % missing – not directly useful for tree models
        'End_Lat', 'End_Lng',
        # near-zero variance
        'Turning_Loop',
        # redundant with one-hot country (all US)
        'Country',
        # duplicate time features already extracted by create_temporal_features
        'Hour', 'DayOfWeek', 'Month', 'Duration_min',
        # Wind_Chill has 26 % missing and is highly correlated with Temperature
        'Wind_Chill(F)',
    ]
    df = df.drop(columns=drop_cols, errors='ignore')

    # ── Step 6: fill missing numerics with median ─────────────────────────────
    numeric_cols = df.select_dtypes(include='number').columns
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())

    # ── Convert bool → int ───────────────────────────────────────────────────
    bool_cols = df.select_dtypes(include='bool').columns
    df[bool_cols] = df[bool_cols].astype(int)

    # ── Step 7: label-encode high-cardinality categoricals ───────────────────
    le = LabelEncoder()
    for col in ['Weather_Condition', 'weather_group']:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown')
            df[col] = le.fit_transform(df[col].astype(str))

    # ── Step 8: one-hot encode low-cardinality categoricals ──────────────────
    low_card_cats = [
        'Source', 'State', 'Timezone', 'Wind_Direction',
        'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight',
        'Astronomical_Twilight',
    ]
    for col in low_card_cats:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown')
    df = pd.get_dummies(
        df,
        columns=[c for c in low_card_cats if c in df.columns],
        drop_first=True,
    )

    # ── Ensure Severity is plain int ─────────────────────────────────────────
    df['Severity'] = df['Severity'].astype(int)

    # ── Step 9: scale continuous numeric features ─────────────────────────────
    binary_cols = [c for c in df.select_dtypes(include='number').columns
                   if df[c].nunique() <= 2]
    scale_cols = [c for c in df.select_dtypes(include='number').columns
                  if c != 'Severity' and c not in binary_cols]
    scaler = StandardScaler()
    df[scale_cols] = scaler.fit_transform(df[scale_cols])

    return df


# ── Build & verify ────────────────────────────────────────────────────────────
df_ml = build_ml_ready_df('city_traffic_accidents.csv')

assert isinstance(df_ml, pd.DataFrame), "Should return a DataFrame"
assert df_ml.shape[0] == 500_000, f"Expected 500 000 rows, got {df_ml.shape[0]}"
assert df_ml.isnull().sum().sum() == 0, "No missing values expected"
assert df_ml.select_dtypes(include='object').shape[1] == 0, "All columns should be numeric"
assert 'Severity' in df_ml.columns, "Target column Severity must be present"
assert df_ml['Severity'].isin([1, 2, 3, 4]).all(), "Severity must be 1-4"
assert any(c.startswith('State_') for c in df_ml.columns), "State should be one-hot encoded"

print("[PASS] Final ML-ready DataFrame built successfully!")
print(f"  Shape            : {df_ml.shape[0]:,} rows × {df_ml.shape[1]} columns")
print(f"  Missing values   : {df_ml.isnull().sum().sum()}")
print(f"  Dtypes           : {df_ml.dtypes.value_counts().to_dict()}")
print(f"  Severity classes : {sorted(df_ml['Severity'].unique())}")
print()
print("Feature groups:")
print(f"  Temporal         : hour, day_of_week, month, is_weekend, is_rush_hour, duration_min …")
print(f"  Weather          : Temperature(F), Visibility(mi), is_freezing, low_visibility, weather_group …")
print(f"  Road             : n_road_features, has_traffic_control, Junction, Crossing …")
print(f"  Encoded cats     : {sum(1 for c in df_ml.columns if c.startswith('State_'))} State cols, "
      f"{sum(1 for c in df_ml.columns if c.startswith('Wind_Direction_'))} Wind_Direction cols …")
print()
print("df_ml is ready for train/test split → model training.")


[PASS] Final ML-ready DataFrame built successfully!
  Shape            : 500,000 rows × 125 columns
  Missing values   : 0
  Dtypes           : {dtype('bool'): 86, dtype('int64'): 23, dtype('float64'): 16}
  Severity classes : [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Feature groups:
  Temporal         : hour, day_of_week, month, is_weekend, is_rush_hour, duration_min …
  Weather          : Temperature(F), Visibility(mi), is_freezing, low_visibility, weather_group …
  Road             : n_road_features, has_traffic_control, Junction, Crossing …
  Encoded cats     : 48 State cols, 24 Wind_Direction cols …

df_ml is ready for train/test split → model training.


Reminder: Use class_weight='balanced' on all classifiers — Severity 2 is 80% of the data

In [ ]:
df_ml.head(10)

,Severity,Start_Lat,Start_Lng,Distance(mi),Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Wind_Chill_missing,Precipitation_missing,hour,day_of_week,month,is_weekend,is_morning_rush,is_evening_rush,is_rush_hour,duration_min,weather_data_available,is_freezing,low_visibility,weather_group,n_road_features,has_traffic_control,Source_Source2,Source_Source3,State_AR,State_AZ,State_CA,State_CO,State_CT,State_DC,State_DE,State_FL,State_GA,State_IA,State_ID,State_IL,State_IN,State_KS,State_KY,State_LA,State_MA,State_MD,State_ME,State_MI,State_MN,State_MO,State_MS,State_MT,State_NC,State_ND,State_NE,State_NH,State_NJ,State_NM,State_NV,State_NY,State_OH,State_OK,State_OR,State_PA,State_RI,State_SC,State_SD,State_TN,State_TX,State_UT,State_VA,State_VT,State_WA,State_WI,State_WV,State_WY,Timezone_US/Eastern,Timezone_US/Mountain,Timezone_US/Pacific,Timezone_Unknown,Wind_Direction_Calm,Wind_Direction_E,Wind_Direction_ENE,Wind_Direction_ESE,Wind_Direction_East,Wind_Direction_N,Wind_Direction_NE,Wind_Direction_NNE,Wind_Direction_NNW,Wind_Direction_NW,Wind_Direction_North,Wind_Direction_S,Wind_Direction_SE,Wind_Direction_SSE,Wind_Direction_SSW,Wind_Direction_SW,Wind_Direction_South,Wind_Direction_Unknown,Wind_Direction_VAR,Wind_Direction_Variable,Wind_Direction_W,Wind_Direction_WNW,Wind_Direction_WSW,Wind_Direction_West,Sunrise_Sunset_Night,Sunrise_Sunset_Unknown,Civil_Twilight_Night,Civil_Twilight_Unknown,Nautical_Twilight_Night,Nautical_Twilight_Unknown,Astronomical_Twilight_Night,Astronomical_Twilight_Unknown
0,2,-0.073010,0.924328,-0.324662,0.387666,0.360523,0.227519,0.332156,-0.518556,-0.067044,0.989264,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.128679,-0.939231,0.934671,0,0,0,0,0.122936,1,0,0,-0.041268,-0.570460,0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,2,-0.022944,-0.075065,-0.324662,0.759777,0.891538,-0.533144,0.332156,0.071103,0.045776,0.448353,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1.217399,-0.355582,0.934671,0,0,0,0,-0.541639,1,0,0,1.861167,-0.570460,0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,True,False,False,False
2,2,-0.525612,0.455159,-0.236280,1.450841,-0.834263,-0.212865,0.332156,-0.911662,-0.067044,-0.705591,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0.320976,1.979017,0.354698,1,0,0,0,-0.014042,1,0,0,-0.675413,0.791677,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
3,2,-0.420294,-1.317670,0.180624,1.185047,-1.453781,-0.703293,0.332156,0.267657,-0.067044,-0.705591,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.282460,0.811718,-0.225275,0,0,1,1,-0.127107

In [ ]:
df_ml.info(140)

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 125 columns):
 #    Column                         Dtype  
---   ------                         -----  
 0    Severity                       int64  
 1    Start_Lat                      float64
 2    Start_Lng                      float64
 3    Distance(mi)                   float64
 4    Temperature(F)                 float64
 5    Humidity(%)                    float64
 6    Pressure(in)                   float64
 7    Visibility(mi)                 float64
 8    Wind_Speed(mph)                float64
 9    Precipitation(in)              float64
 10   Weather_Condition              float64
 11   Amenity                        int64  
 12   Bump                           int64  
 13   Crossing                       int64  
 14   Give_Way                       int64  
 15   Junction                       int64  
 16   No_Exit                        int64  
 17   Railway                        int64  